In [1]:
"""PCA decision project: Olivetti faces (400 x 4096), task = 40-class identity classification."""
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_olivetti_faces
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.manifold import TSNE, trustworthiness
from sklearn.neighbors import NearestNeighbors

In [2]:

SEED = 42
rng = np.random.default_rng(SEED)
X, y = fetch_olivetti_faces(return_X_y=True, random_state=SEED)
n, d = X.shape
print(f"R1 data: Olivetti faces (sklearn/AT&T), n={n}, d={d}, task: 40-class identity, LogisticRegression")
Xs = StandardScaler().fit_transform(X)


downloading Olivetti faces from https://ndownloader.figshare.com/files/5976027 to /root/scikit_learn_data
R1 data: Olivetti faces (sklearn/AT&T), n=400, d=4096, task: 40-class identity, LogisticRegression


In [3]:
# ---------------- R2 scree + noise control ----------------
def scree(A):
    p = PCA().fit(A); c = np.cumsum(p.explained_variance_ratio_)
    return {f"k@{int(q*100)}%": int(np.searchsorted(c, q) + 1) for q in (.80, .90, .95, .99)}, p.explained_variance_ratio_[:5]
noise = StandardScaler().fit_transform(rng.normal(size=X.shape))
for name, A in [("faces", Xs), ("noise", noise)]:
    s, r5 = scree(A); print("R2", name, s, "first5:", np.round(r5, 4))

R2 faces {'k@80%': 26, 'k@90%': 66, 'k@95%': 123, 'k@99%': 260} first5: [0.2688 0.1234 0.0784 0.0469 0.0325]
R2 noise {'k@80%': 277, 'k@90%': 331, 'k@95%': 363, 'k@99%': 391} first5: [0.0043 0.0042 0.0042 0.0042 0.0042]


In [4]:
# ---------------- R3 own PCA vs sklearn ----------------
def pca_fit(A, k):
    mu = A.mean(0); Ac = A - mu
    C = Ac.T @ Ac / (len(A) - 1)
    w, V = np.linalg.eigh(C); o = np.argsort(w)[::-1]
    return mu, w[o][:k], V[:, o][:, :k]
k3 = 50
mu, w, V = pca_fit(Xs, k3); sk = PCA(k3, svd_solver="full").fit(Xs)
sgn = np.sign(np.sum(V.T * sk.components_, axis=1)); V = V * sgn
Z_own, Z_sk = (Xs - mu) @ V, sk.transform(Xs)
print("R3 components  :", np.allclose(V.T, sk.components_, atol=1e-6))
print("R3 exp. var    :", np.allclose(w, sk.explained_variance_, atol=1e-8))
print("R3 projections :", np.allclose(Z_own, Z_sk, atol=1e-6))
print("R3 reconstruct :", np.allclose(Z_own @ V.T + mu, sk.inverse_transform(Z_sk), atol=1e-6))

R3 components  : False
R3 exp. var    : True
R3 projections : False
R3 reconstruct : False


In [5]:
# ---------------- R4 cost of reduction (everything inside Pipeline) ----------------
def cv_scores(step, scale=True, seed=SEED):
    steps = ([("sc", StandardScaler())] if scale else []) + ([("red", step)] if step else []) \
            + [("clf", LogisticRegression(max_iter=3000))]
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    return cross_val_score(Pipeline(steps), X, y, cv=cv)
K_FULL = int(n * 0.8) - 1                        # rank of a training fold after centring (320-1); k=d is impossible when d>n
rows = []
for label, step in [("none", None)] + [(f"PCA k={k}", PCA(k, random_state=SEED)) for k in (10, 25, 50, 100, 200, K_FULL)]:
    s = cv_scores(step); rows.append((label, s.mean(), s.std(), s.mean() * n))
t4 = pd.DataFrame(rows, columns=["model", "acc", "fold_std", "rows_correct_of_400"])
t4["gap_vs_none_rows"] = t4.rows_correct_of_400 - t4.rows_correct_of_400.iloc[0]
print("R4\n", t4.round(4).to_string(index=False))


R4
     model    acc  fold_std  rows_correct_of_400  gap_vs_none_rows
     none 0.9750    0.0079                390.0               0.0
 PCA k=10 0.8925    0.0376                357.0             -33.0
 PCA k=25 0.9650    0.0146                386.0              -4.0
 PCA k=50 0.9800    0.0061                392.0               2.0
PCA k=100 0.9800    0.0061                392.0               2.0
PCA k=200 0.9775    0.0094                391.0               1.0
PCA k=319 0.9775    0.0094                391.0               1.0


In [6]:
# ---------------- R5 extraction vs selection ----------------
r5 = []
for k in (25, 50, 100):
    r5.append((k, cv_scores(PCA(k, random_state=SEED)).mean(),
               cv_scores(SelectKBest(f_classif, k=k)).mean()))
print("R5\n", pd.DataFrame(r5, columns=["budget", "PCA", "SelectKBest"]).round(4).to_string(index=False))


R5
  budget   PCA  SelectKBest
     25 0.965       0.4125
     50 0.980       0.5250
    100 0.980       0.7575


In [7]:
# ---------------- R6 2-D picture ----------------
def overlap(A, Z, k=10):
    ia = NearestNeighbors(n_neighbors=k + 1).fit(A).kneighbors(A, return_distance=False)[:, 1:]
    iz = NearestNeighbors(n_neighbors=k + 1).fit(Z).kneighbors(Z, return_distance=False)[:, 1:]
    return np.mean([len(set(a) & set(b)) / k for a, b in zip(ia, iz)]), iz
Zp = PCA(2, random_state=SEED).fit_transform(Xs)
tp, (op, _) = trustworthiness(Xs, Zp, n_neighbors=10), overlap(Xs, Zp)
print(f"R6 PCA-2D: trustworthiness={tp:.3f} kNN-overlap={op:.3f}")
embs, nbrs = {}, {}
for perp in (5, 15, 30, 50):
    Z = TSNE(2, perplexity=perp, init="pca", random_state=SEED).fit_transform(Xs)
    o, iz = overlap(Xs, Z); embs[perp], nbrs[perp] = Z, iz
    print(f"R6 t-SNE perp={perp}: trustworthiness={trustworthiness(Xs, Z, n_neighbors=10):.3f} kNN-overlap={o:.3f}")
ps = list(nbrs); st = [np.mean([len(set(a) & set(b)) / 10 for a, b in zip(nbrs[ps[i]], nbrs[ps[j]])])
                       for i in range(4) for j in range(i + 1, 4)]
print(f"R6 t-SNE stability (neighbour overlap between perplexities): mean={np.mean(st):.3f} min={np.min(st):.3f}")
CHOSEN = "tsne" if max(trustworthiness(Xs, embs[30], n_neighbors=10), 0) > tp else "pca"   # edit after reading numbers
Zf, ttl = (embs[30], "t-SNE (perplexity 30)") if CHOSEN == "tsne" else (Zp, "PCA")
fig, ax = plt.subplots(figsize=(6, 5)); sc = ax.scatter(Zf[:, 0], Zf[:, 1], c=y, cmap="nipy_spectral", s=12)
ax.set_xlabel(f"{ttl} axis 1"); ax.set_ylabel(f"{ttl} axis 2"); ax.set_title("Olivetti faces, coloured by identity")
fig.savefig("final_figure.png", dpi=200, bbox_inches="tight")

R6 PCA-2D: trustworthiness=0.860 kNN-overlap=0.280
R6 t-SNE perp=5: trustworthiness=0.951 kNN-overlap=0.603
R6 t-SNE perp=15: trustworthiness=0.952 kNN-overlap=0.599
R6 t-SNE perp=30: trustworthiness=0.940 kNN-overlap=0.588
R6 t-SNE perp=50: trustworthiness=0.930 kNN-overlap=0.569
R6 t-SNE stability (neighbour overlap between perplexities): mean=0.670 min=0.626


In [8]:
# ---------------- R8 robustness ----------------
print("R8a CV seed change (PCA k=50 vs none):")
for sd in (0, 1, 2):
    print(f"  seed {sd}: none={cv_scores(None, seed=sd).mean():.4f}  PCA50={cv_scores(PCA(50, random_state=SEED), seed=sd).mean():.4f}")
s_raw, r_raw = scree(X)
print("R8b no standardisation: scree", s_raw, "first5:", np.round(r_raw, 4))
print(f"  CV none={cv_scores(None, scale=False).mean():.4f}  PCA50={cv_scores(PCA(50, random_state=SEED), scale=False).mean():.4f}")

R8a CV seed change (PCA k=50 vs none):
  seed 0: none=0.9750  PCA50=0.9825
  seed 1: none=0.9625  PCA50=0.9750
  seed 2: none=0.9750  PCA50=0.9775
R8b no standardisation: scree {'k@80%': 27, 'k@90%': 66, 'k@95%': 123, 'k@99%': 260} first5: [0.2381 0.1399 0.0797 0.05   0.0361]
  CV none=0.9800  PCA50=0.9800
